In [9]:
# Question 1
# Weekly Assessment -Agentic AI

# Description
# Capstone Assessment (Duration: 2 Hours)

# 🔷 Scenario 1: AI Customer Support Multi-Agent System

# 🎯 Objective:
# Build a multi-agent customer support system that can classify, delegate, and resolve user queries.

# 💼 Problem Statement
# A company wants to automate its support system using AI agents.

# The system should:

# Understand customer queries
# Route them to the correct department
# Provide responses

# 👥 Agents to Build

# Classifier Agent
# Identifies query type (Billing / Technical / General)

# Billing Agent
# Handles payment/refund queries

# Technical Support Agent
# Handles app/software issues

# Response Agent
# Combines outputs into a final response

# ⚙️ Tasks to Implement

# Multi-agent architecture (at least 3 agents mandatory)
# Task delegation logic
# Basic agent communication (message passing / function calls)

# 💡 Sample Inputs

# “I was charged twice for my subscription”
# “The app crashes when I open it”
# “What are your pricing plans?”

# 🧪 Expected Output

# Correct classification of query
# Delegation to appropriate agent
# Final structured response

# 🧰 Tools (Choose Any)

# Python (basic functions or LLM APIs)
# LangGraph (workflow)
# CrewAI (role-based agents)
# AutoGen (agent conversation)
# Microsoft Copilot Studio (optional low-code)

# 📦 Deliverables

# Working code / flow
# Architecture diagram
# 2–3 test case outputs

# ⏱️ Suggested Time Split

# Design: 20 min
# Implementation: 70 min
# Testing & documentation: 30 min




                                     #Architecture

                              # User Query
                              #     ↓
                              # Classifier Agent
                              #     ↓
                              # ┌───────────────┬───────────────┬───────────────┐
                              # ↓               ↓               ↓
                              # Billing Agent   Technical Agent  General Agent
                              #     ↓               ↓               ↓
                              #         Response Agent (Final Output)




!pip install openai gradio

# SECURE API KEY SETUP
import os
from openai import OpenAI

# OPTION 1: (Colab users)
try:
    from google.colab import userdata
    api_key = userdata.get("GROQ_API_KEY")
except:
    api_key = os.getenv("GROQ_API_KEY")

# Safety check
if not api_key:
    raise ValueError(" API key not found. Set GROQ_API_KEY securely.")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "llama-3.3-70b-versatile"

# DATABASE (MEMORY)
import sqlite3

conn = sqlite3.connect("memory.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    query TEXT,
    category TEXT,
    response TEXT
)
""")
conn.commit()

def save_memory(query, category, response):
    cursor.execute(
        "INSERT INTO history (query, category, response) VALUES (?, ?, ?)",
        (query, category, response)
    )
    conn.commit()

def get_memory(limit=3):
    cursor.execute(
        "SELECT query FROM history ORDER BY id DESC LIMIT ?", (limit,)
    )
    return [row[0] for row in cursor.fetchall()]

# LLM HELPER FUNCTION

def llm_response(prompt):
    try:
        res = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}]
        )
        return res.choices[0].message.content.strip()
    except Exception as e:
        return f"⚠️ Error: {str(e)}"

# AGENTS

class ClassifierAgent:
    def run(self, query, context):
        prompt = f"""
You are a classifier agent.

Previous queries: {context}

Classify the user query into:
- Billing
- Technical
- General

Query: {query}

Return only one word.
"""
        return llm_response(prompt)


class BillingAgent:
    def run(self, query):
        return llm_response(f"Resolve this billing issue professionally: {query}")


class TechnicalAgent:
    def run(self, query):
        return llm_response(f"Provide troubleshooting steps: {query}")


class GeneralAgent:
    def run(self, query):
        return llm_response(f"Answer this general question clearly: {query}")


class ResponseAgent:
    def run(self, category, response, context):
        return f"""
🧠 Previous Queries: {context}

🔍 Category: {category}

✅ Final Response:
{response}
"""

# MULTI-AGENT SYSTEM

class SupportSystem:
    def __init__(self):
        self.classifier = ClassifierAgent()
        self.billing = BillingAgent()
        self.technical = TechnicalAgent()
        self.general = GeneralAgent()
        self.response = ResponseAgent()

    def handle_query(self, query):
        context = get_memory()

        # Step 1: Classification
        category = self.classifier.run(query, context)

        category = category.lower()

        # Step 2: Delegation

        message = f"Category: {category}, Query: {query}"
        if "Billing" in category:
            result = self.billing.run(message)
        elif "Technical" in category:
            result = self.technical.run(message)
        else:
           result = self.general.run(message)

        # Step 3: Final Response
        final_response = self.response.run(category, result, context)

        # Step 4: Save memory
        save_memory(query, category, final_response)

        return final_response

# Test case

system = SupportSystem()

test_queries = [
    "I was charged twice for my subscription",
    "App crashes when I open it",
    "What are your pricing plans?"
]

for q in test_queries:
    print("USER:", q)
    print(system.handle_query(q))
    print("="*50)


# GRADIO UI (BONUS)


import gradio as gr

def chatbot(query):
    return system.handle_query(query)

ui = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs="text",
    title="🤖 AI Customer Support Multi-Agent System"
)

ui.launch()

USER: I was charged twice for my subscription

🧠 Previous Queries: ['What are your pricing plans?', 'App crashes when I open it', 'I was charged twice for my subscription']

🔍 Category: billing

✅ Final Response:
If you were charged twice for your subscription, it's likely an error on the part of the billing system or company. Here are some steps you can take to resolve the issue:

1. **Contact the company's customer support**: Reach out to the company's customer support via phone, email, or live chat, and explain the situation. They should be able to investigate and correct the error.
2. **Provide proof of the duplicate charge**: Be prepared to provide proof of the duplicate charge, such as a copy of your bank statement or a screenshot of the transaction.
3. **Request a refund**: Ask the company to refund the duplicate charge as soon as possible.
4. **Check your account settings**: Ensure that your account settings are correct and that you don't have multiple active subscriptions.
5. 